In [1]:
import torch
import torch.nn.functional as funcy

In [6]:
def ssim(i1,i2,wind=11,c1=0.01**2,c2=0.03**2):
    def gwin(wind,sigma=1.5):
        coords = torch.arange(wind) - wind // 2
        g = torch.exp(-(coords**2) / (2 * sigma**2))
        g = g / g.sum()
        return g.unsqueeze(1) @ g.unsqueeze(0)
    window = gwin(wind).to(i1.device)
    window = window.expand(i1.size(1), 1, wind, wind)
    mu1 = funcy.conv2d(i1, window, padding=wind//2, groups=i1.size(1))
    mu2 = funcy.conv2d(i2, window, padding=wind//2, groups=i2.size(1))
    mu1_sq = mu1 ** 2
    mu2_sq = mu2 ** 2
    mu1_mu2 = mu1 * mu2
    sigma1_sq = funcy.conv2d(i1**2, window, padding=wind//2, groups=i1.size(1)) - mu1_sq
    sigma2_sq = funcy.conv2d(i2**2, window, padding=wind//2, groups=i2.size(1)) - mu2_sq
    sigma12 = funcy.conv2d(i1 * img2, window, padding=wind//2, groups=i1.size(1)) - mu1_mu2
    ssim_map = ((2 * mu1_mu2 + c1) * (2 * sigma12 + c2)) / \
               ((mu1_sq + mu2_sq + c1) * (sigma1_sq + sigma2_sq + c2))

    return ssim_map.mean()

In [7]:
img1 = torch.rand(1, 1, 256, 256)  # example
img2 = torch.rand(1, 1, 256, 256)

score = ssim(img1, img2)
print(score.item())


0.01319311372935772
